In [344]:
import pandas as pd
from generate_xml import load_data
from lxml import etree

In [345]:
data,metadata = load_data('../../zaebuc_written/ZAEBUC-v2.0_release/')
data.head()

/Users/f/Library/CloudStorage/SynologyDrive-ba3sasah/camelLab/zaebuc/corpus_app/src/generate_xml.py:146: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  en = pd.read_csv(f'{datadir}corrected.analyzed_en.tsv',sep='\t',index_col=0)


word flag  ... core_pgn pron_pgn
doc_id         Line_Index idx                     ...                  
en-2019-116710 1.0        1    Developments  NaN  ...      NaN      NaN
                          2              in  NaN  ...      NaN      NaN
                          3             the  NaN  ...      NaN      NaN
                          4             UAE  NaN  ...      NaN      NaN
                          5               ,  NaN  ...      NaN      NaN

[5 rows x 13 columns]

In [394]:
from lxml import etree
import re
namespaces = {'xml': 'http://www.w3.org/XML/1998/namespace'}

corpus = etree.Element("corpus")

for doc_id in metadata.index[:]:
# for doc_id in ['ar-2021-X32635', 'ar-2021-X32635']:
    doc = metadata.loc[doc_id].to_frame().drop('text').T.reset_index(names='idx')
    doc['textDirection'] = doc['language'].map(lambda x: 'rtl' if x=='Arabic' else 'ltr')
    # doc['idx'] = doc['doc_id']
    doc = doc.to_xml(root_name='corpus',row_name='doc',attr_cols=doc.columns.to_list(),xml_declaration=False,index=False)
    doc = etree.fromstring(doc)

    linesidxs = data.loc[doc_id].index.get_level_values(0).drop_duplicates()
    
    for lineidx in linesidxs:        
        line = etree.Element('sent')
        line.attrib['idx'] = str(int(lineidx))        
        doc[0].append(line)
        
        words = data.loc[(doc_id,lineidx)][:]
        
        # words = words.to_xml(root_name='wordroot',row_name='word',attr_cols=['idx'],elem_cols=words.columns.to_list()[1:],index=False,xml_declaration=False)
        
        # words = etree.Element('wordsss')
        
        for idx in words.index:
            if words.loc[idx,'manual_pos'] == 'PUNCT':
                word = etree.Element('punct')
                word.attrib[f'{{{namespaces["xml"]}}}id'] = f'w.wx.{idx}'            
                word.text = words.loc[idx,'word']
                line.append(word)
                continue
            else:                
                word = etree.Element('word')
                
            word.attrib['idx'] = str(idx)            
            word.attrib[f'{{{namespaces["xml"]}}}id'] = f'w.wx.{idx}'    

            if 'en'  in doc_id: #english entries don't have gloss, take lemma
                gloss_value = words.loc[idx,'manual_lemma'].replace('+','')
                gloss_element = etree.Element('gloss')
                gloss_element.set('class',gloss_value)
                word.append(gloss_element)
                
            for analysisid in words.columns:
                value = words.loc[idx, analysisid]
                if pd.isna(value):
                    continue
                if analysisid == 'word':
                    analysis = etree.Element('text')
                    analysis.text = value
                elif analysisid == 'gloss':
                    
                    full_gloss_element = etree.Element("gloss_full")
                                        
                    full_gloss_element.set('class', value)
                    word.append(full_gloss_element)
                    glosses = re.split(r',|;', value)
                    
                    for gloss in set(glosses):
                        gloss = gloss.strip()
                        gloss_element = etree.Element("gloss")
                        gloss_element.set('class', gloss)
                        word.append(gloss_element)
                        continue
                
                else:
                    analysis = etree.Element(analysisid) 
                    analysis.attrib['class'] = value
                
                word.append(analysis)    
                
            line.append(word)
            
    corpus.append(doc[0])

etree.indent(corpus, space="    ")
tree = etree.ElementTree(corpus)
tree.write("../data/zaebuc_written.xml", pretty_print=True, xml_declaration=True, encoding="utf-8")
# p(corpus)

/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_13595/434697319.py:22: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_13595/434697319.py:22: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_13595/434697319.py:22: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_13595/434697319.py:22: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_13595/434697319.py:22: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl

In [381]:
metadata.columns.to_list()

['text',
 'language',
 'writer_id',
 'year',
 'course',
 'word_count',
 'cefr_avg',
 'major',
 'school_type',
 'school_language',
 'gender',
 'topic',
 'college',
 'residence',
 'writing_mins',
 'handwritten',
 'earlier_task_language',
 'days_between_tasks',
 'safe_assign_score',
 'cefr_1',
 'cefr_2',
 'cefr_3',
 'split']

In [359]:
data.loc[(doc_id, lineidx)][:]

/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_13595/1918117432.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  data.loc[(doc_id, lineidx)][:]


,word,flag,auto_tokenization,auto_pos,auto_lemma,manual_tokenization,manual_pos,manual_lemma,comment,manual_diacritized_lemma,gloss,core_pgn,pron_pgn
idx,,,,,,,,,,,,,
3,تعتبر,NaN,تعتبر,VERB,اعتبر,تعتبر,VERB,اعتبر,NaN,اِعْتَبَر,believe; be regarded; consider; be considered;...,3fs,###
4,ثقافة,NaN,ثقافة,NOUN,ثقافة,ثقافة,NOUN,ثقافة,NaN,ثَقافَة,culture; civilization,###,###
5,التسامح,NaN,التسامح,NOUN,تسامح,التسامح,NOUN,تسامح,NaN,تَسامُح,tolerance,###,###
6,من,NaN,من,ADP,من,من,ADP,من,NaN,مِن,from,###,###
7,أهم,NaN,أهم,ADJ,أهم,أهم,ADJ,أهم,NaN,أَهَمّ,more/most important,###,###
...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,ويسامح,NaN,و+يسامح,CCONJ+VERB,سامح,و+يسامح,CCONJ+VERB,سامح,NaN,سامَح,treat kindly; pardon; forgive,3ms,###
144,ويعفو,NaN,و+يعفو,CCONJ+VERB,عفا,و+يعفو,CCONJ+VERB,عفا,NaN,عَفا,be excused; excuse; be forgiven; forgive,3ms,###
145,عند,NaN,عند,NOUN,عند,عند,NOUN,عند,NaN,عِنْد,with/at,###,###


In [302]:
import lxml.etree.ElementTree

ModuleNotFoundError: No module named 'lxml.etree.ElementTree'; 'lxml.etree' is not a package

In [ ]:
import lxml.etree.ElementTree as ET

# ET.register_namespace(namespaces)
etree.register_namespace(namespaces)

from io import BytesIO

f = BytesIO()

tree = ET.ElementTree(corpus)
tree.register_namspace("xml",'http://www.w3.org/XML/1998/namespace')
tree.write(f, encoding='utf-8', xml_declaration=True)


print(f.getvalue().decode('utf-8'))

ModuleNotFoundError: No module named 'lxml.etree.ElementTree'; 'lxml.etree' is not a package

In [299]:
ET.ElementTree(doc)

In [180]:
data[data.index.get_level_values(1)==2]

Word Flag  ... Core_PGN Pron_PGN
doc_id         Line_Index idx                ...                  
en-2019-116710 2.0        24       The  NaN  ...      NaN      NaN
                          25       UAE  NaN  ...      NaN      NaN
                          26       has  NaN  ...      NaN      NaN
                          27         a  NaN  ...      NaN      NaN
                          28    desert  NaN  ...      NaN      NaN
...                                ...  ...  ...      ...      ...
ar-2021-X32635 2.0        183   للتعدي  NaN  ...      ###      ###
                          184      على  NaN  ...      ###      ###
                          185  خصوصيات  NaN  ...      ###      ###
                          186    غيرهم  NaN  ...      ###      3mp
                          187        .  NaN  ...      ###      ###

[44155 rows x 13 columns]

In [290]:
def p(xml): 
    return print(str(etree.tostring(xml, pretty_print=True,encoding='utf-8',xml_declaration=True).decode('utf-8')))

In [283]:
p(words)

TypeError: tostring() got an unexpected keyword argument 'resolve_entities'